In [55]:

# imports
import json
import pandas as pd
from os import path

# constants
intersection_data_path = ("~/code/county_coverage/data/raw/intersections/" +
                            "Jefferson_County_KY_Street_Intersections.geojson")

In [191]:
def get_records(intersection_data_path):
    with open(idp, 'r') as file:
        data = json.load(file)
    features = data['features']
    for feature in features:
        properties = feature['properties']
        geometry = feature['geometry']
        # properties['geo_type'] = geometry['type'] # Always == "Point". Not useful.
        properties['GEOMETRY'] = geometry['coordinates']
        yield properties

df = pd.DataFrame.from_dict(get_records(path.expanduser(intersection_data_path))).convert_dtypes()

bad_row = 100354
# see notes below
# full of nulls that mess up road name compression
# remove row before next step

df = df.drop(df[df.OBJECTID == bad_row].index)

df.head()

,OBJECTID,SIFCODE1,SIFCODE2,INTID,SCCAD_ID,FST_INTPRE,FST_INTNAME,FST_INTSUF,SEC_INTPRE,SEC_INTNAME,SEC_INTSUF,X_COORD,Y_COORD,FST_SIFID,SEC_SIFID,GLOBALID,GEOMETRY
0,1,5464,7662,154647662,1,,REHL,RD,W,REHL,CT,1278243.0,259531.9375,4976,6856,{CD0D9D51-41FB-46FF-B229-AC9C0DDB7E61},"[-85.51044384084946, 38.20588660809318]"
1,2,5464,6551,254646551,2,,REHL,RD,,TUCKER STATION,RD,1273126.375,257588.25,4976,5908,{A9FAED81-F7AE-436F-A657-F95FE1B905E8},"[-85.52816882852979, 38.20037561255241]"
2,3,5464,6551,354646551,3,,REHL,RD,,TUCKER STATION,RD,1273050.50875,257590.85375,4976,5908,{71155AF3-3487-4EDA-8B72-B282378DE7F3},"[-85.52841872335158, 38.200360349057064]"
3,4,3194,9996,431949996,4,,I 64 EAST,,,I 265 RAMP,,1279903.25,265597.5,3076,8763,{AD332CAB-27B0-48B7-ACBF-5CDEEC31654B},"[-85.50495414554669, 38.22260344055154]"
4,5,9349,9996,593499996,5,,I 265 NORTH,,,I 265 RAMP,,1279730.75,265426.4375,8197,8763,{0FE38BAB-8B39-4BE8-BA57-1DAA46FDD4BC},"[-85.50554642815675, 38.222127288727606]"


In [193]:
## find a good index for data

## SCCAD_ID -> No
df.SCCAD_ID.is_unique # False
vc = df.SCCAD_ID.value_counts()
sc2 = vc[vc > 1].index # SCCAD_ID s that point to more than one intersection

df[df.SCCAD_ID.isin(sc2)]


## OBJECTID
df.OBJECTID.is_unique # True


## GLOBALID

  # annoying to look at and use:
  # strings like this: {CD0D9D51-41FB-46FF-B229-AC9C0DDB7E61}

df.GLOBALID.is_unique # True
df.GLOBALID.apply(hash) # negative numbers


## INTID ->  best choice

 # derived from SCCAD_ID + SIFCODES 1 and 2

df.INTID.is_unique # True: can use as index?

df.INTID.str.strip().is_unique # still true

df.INTID.apply(hash) # includes negative numbers
set(''.join(df.INTID.str.strip())) # ->
 # {'0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'C', 'D', 'E', 'F'}
 # only hex chars

 # INTID is a string but it cen be expressed as a number.

df.INTID.apply(lambda x:int(x, base=16)).is_unique # True

"""Conclusion: 
  drop OBJECTID, GLOBALID, SCCAD_ID
  convert INTID to integer and use the result as index"""

df1 = df.drop(['OBJECTID', 'SCCAD_ID', 'GLOBALID'], axis=1)
df1.INTID = df1.INTID.apply(lambda x:int(x, base=16))
df1 = df1.set_index('INTID')
df1.head()

,SIFCODE1,SIFCODE2,FST_INTPRE,FST_INTNAME,FST_INTSUF,SEC_INTPRE,SEC_INTNAME,SEC_INTSUF,X_COORD,Y_COORD,FST_SIFID,SEC_SIFID,GEOMETRY
INTID,,,,,,,,,,,,,
5710837346,5464,7662,,REHL,RD,W,REHL,CT,1278243.0,259531.9375,4976,6856,"[-85.51044384084946, 38.20588660809318]"
10005800273,5464,6551,,REHL,RD,,TUCKER STATION,RD,1273126.375,257588.25,4976,5908,"[-85.52816882852979, 38.20037561255241]"
14300767569,5464,6551,,REHL,RD,,TUCKER STATION,RD,1273050.50875,257590.85375,4976,5908,"[-85.52841872335158, 38.200360349057064]"
18011691414,3194,9996,,I 64 EAST,,,I 265 RAMP,,1279903.25,265597.5,3076,8763,"[-85.50495414554669, 38.22260344055154]"
23945910678,9349,9996,,I 265 NORTH,,,I 265 RAMP,,1279730.75,265426.4375,8197,8763,"[-85.50554642815675, 38.222127288727606]"


In [189]:
# Compress road name info
fst_road_info = df1[["FST_INTPRE", "FST_INTNAME", "FST_INTSUF"]].apply(" ".join, axis=1).str.strip()
sec_road_info = df1[["SEC_INTPRE", "SEC_INTNAME", "SEC_INTSUF"]].apply(" ".join, axis=1).str.strip()

intersections = df1.drop(["FST_INTPRE", "FST_INTNAME", "FST_INTSUF", 
                          "SEC_INTPRE", "SEC_INTNAME", "SEC_INTSUF"], axis=1)
intersections['FST_ROADNAME'] = fst_road_info
intersections['SEC_ROADNAME'] = sec_road_info
intersections.head()


,SIFCODE1,SIFCODE2,X_COORD,Y_COORD,FST_SIFID,SEC_SIFID,GEOMETRY,FST_ROADNAME,SEC_ROADNAME
INTID,,,,,,,,,
5710837346,5464,7662,1278243.0,259531.9375,4976,6856,"[-85.51044384084946, 38.20588660809318]",REHL RD,W REHL CT
10005800273,5464,6551,1273126.375,257588.25,4976,5908,"[-85.52816882852979, 38.20037561255241]",REHL RD,TUCKER STATION RD
14300767569,5464,6551,1273050.50875,257590.85375,4976,5908,"[-85.52841872335158, 38.200360349057064]",REHL RD,TUCKER STATION RD
18011691414,3194,9996,1279903.25,265597.5,3076,8763,"[-85.50495414554669, 38.22260344055154]",I 64 EAST,I 265 RAMP
23945910678,9349,9996,1279730.75,265426.4375,8197,8763,"[-85.50554642815675, 38.222127288727606]",I 265 NORTH,I 265 RAMP


Why are FST_SIFID and SEC_SIFID floats on import?

-> because of a NAN value in item where OBJECTID == 100354
bad_row = 100354

both CSV and JSON are like this

#### CSV:
```csv
X,Y,OBJECTID,SIFCODE1,SIFCODE2,INTID,SCCAD_ID,FST_INTPRE,FST_INTNAME,FST_INTSUF,SEC_INTPRE,SEC_INTNAME,SEC_INTSUF,X_COORD,Y_COORD,FST_SIFID,SEC_SIFID,GLOBALID
```

1227948.0,272835.375,100354,,,,,,,,,,,1227948,272835.375,,,{60D18EC3-BBBC-419B-8A10-62C37F39E987}

#### JSON:
```json
null = float('nan')
{ "type": "Feature",
  "properties": {
     "OBJECTID": 100354, "SIFCODE1": null, "SIFCODE2": null, "INTID": null, "SCCAD_ID": null,
     "FST_INTPRE": null, "FST_INTNAME": null, "FST_INTSUF": null, "SEC_INTPRE": null, "SEC_INTNAME": null,
     "SEC_INTSUF": null, "X_COORD": 1227948.0, "Y_COORD": 272835.375, "FST_SIFID": null, "SEC_SIFID": null,
     "GLOBALID": "{60D18EC3-BBBC-419B-8A10-62C37F39E987}" },
      
  "geometry": { "type": "Point", "coordinates": [ -85.686176099576869, 38.240392433657043 ] } }
```

This interferes with some code that simplifies the road names. Remove the row with nulls and any other before further processing. 


In [126]:
# convert coordinates from LOJIC CRS to (longitude, latitude)
# LOJIC projection: ESRI:102679
# NAD_1983_StatePlane_Kentucky_North_FIPS_1601_Feet

# Standard long, lat: epsg:4326

import numpy as np

from pyproj import CRS
from pyproj.transformer import Transformer

KY_grid_CRS = CRS("ESRI:102679")
longlat_CRS = CRS("epsg:4326")

CRS_transformer = Transformer.from_crs(crs_from=KY_grid_CRS, crs_to=longlat_CRS, always_xy=True).transform

#CRS_transformer.transform(1154395.500000, 188677.437500)


In [150]:
# converting grid point to longitude, latitude

#reindex['coordinates'] = 
XYgrid = intersections.X_COORD.combine(df.Y_COORD, lambda x, y:(x, y))

# have to convert this way because CRS_transformer expects 2 arguments
long_lat_coordinates = intersections.X_COORD.combine(intersections.Y_COORD, CRS_transformer)
long_lat_coordinates = long_lat_coordinates.apply(np.array)
long_lat_coordinates


0        [-85.51043903006253, 38.205879314617064]
1         [-85.52814980553048, 38.20034879942594]
2         [-85.52841390771789, 38.20035305747533]
3          [-85.5049493351228, 38.22259614360763]
4         [-85.50554161759499, 38.22211999190267]
                           ...                   
20942     [-85.52218310157899, 38.26474099550489]
20943    [-85.51597235007885, 38.207992191858494]
20944    [-85.51597235007885, 38.207992191858494]
20945    [-85.62589307374952, 38.128086553516376]
20946    [-85.67867380142928, 38.103651860939735]
Length: 20946, dtype: object

In [154]:
geo_diff = (long_lat_coordinates - intersections.GEOMETRY).apply(np.linalg.norm)

intersections.loc[geo_diff[geo_diff > 0.0001].index]

,OBJECTID,SIFCODE1,SIFCODE2,INTID,SCCAD_ID,X_COORD,Y_COORD,FST_SIFID,SEC_SIFID,GEOMETRY,FST_ROADNAME,SEC_ROADNAME
132,134,3792,6777,20237926777,202,1275050.5,270691.5625,12949,6104,"[-85.52219333709645, 38.23642426698946]",LEDGES DR,CREEKVALLEY RD
306,314,0506,2959,46705062959,467,1274759.0,282274.0625,524,2874,"[-85.52377618828518, 38.26818008336028]",BERRYTOWN RD,HINES RD
1278,1306,5322,5736,181353225736,1813,1258051.75,289134.875,4861,5212,"[-85.58213654142249, 38.28602260562131]",CRAWLEY CT,MOCKSHIRE DR
2200,2230,3559,6596,313535596596,3135,1232551.0,266076.1875,3384,13015,"[-85.6695899572017, 38.22201721282951]",KINGS HWY,TYLER LN
4043,4091,D074,D076,5479D074D076,5479,1252004.75,321386.125,10124,10126,"[-85.60501649151297, 38.37496056510169]",CHERRY TREE LN,CHERRY TREE CT
...,...,...,...,...,...,...,...,...,...,...,...,...
19917,63512,0000,F099,290110000F099,29011,1205651.78875,241770.70875,1,14426,"[-85.76193115775621, 38.15421172845835]",NO STREET NAME,KENWOOD BUSINESS DR
19936,64164,F259,7544,29034F2597544,29034,1247722.86625,232348.13125,14758,6763,"[-85.61535931332455, 38.130043022708435]",AVALON SPRINGS DR,ZELMA FIELDS AVE
19937,64165,F259,7544,29035F2597544,29035,1248002.155,232653.7075,14758,6763,"[-85.61453623397385, 38.13091957012409]",AVALON SPRINGS DR,ZELMA FIELDS AVE
20547,89146,F546,9037,29836F5469037,29836,1264045.98625,253670.4325,15341,13486,"[-85.55951097148113, 38.18938192109288]",COLE THOMPSON LN,WILLOWVIEW BLVD


In [194]:

#intersections['GEOMETRY'] = long_lat_coordinates
# could encode this as two columns: longitude and latitude
# might still do it, I will typically use this geometry as a point that gets unpacked
# but it seems annoying to have to access two columns constantly when accessing one 
# and unpacking the value is so easy.
# Mirrors the GEOMETRY column in centerlines as well, which is a list of points
intersections.head()

,SIFCODE1,SIFCODE2,X_COORD,Y_COORD,FST_SIFID,SEC_SIFID,GEOMETRY,FST_ROADNAME,SEC_ROADNAME
INTID,,,,,,,,,
5710837346,5464,7662,1278243.0,259531.9375,4976,6856,"[-85.51044384084946, 38.20588660809318]",REHL RD,W REHL CT
10005800273,5464,6551,1273126.375,257588.25,4976,5908,"[-85.52816882852979, 38.20037561255241]",REHL RD,TUCKER STATION RD
14300767569,5464,6551,1273050.50875,257590.85375,4976,5908,"[-85.52841872335158, 38.200360349057064]",REHL RD,TUCKER STATION RD
18011691414,3194,9996,1279903.25,265597.5,3076,8763,"[-85.50495414554669, 38.22260344055154]",I 64 EAST,I 265 RAMP
23945910678,9349,9996,1279730.75,265426.4375,8197,8763,"[-85.50554642815675, 38.222127288727606]",I 265 NORTH,I 265 RAMP


In [109]:
keep_columns = ["INTID", "FST_ROADNAME", "FST_SIFID",	"SEC_ROADNAME", "SEC_SIFID", "GEOMETRY"]

intersections_clean = intersections[keep_columns].convert_dtypes()
display(intersections_clean.head())


,INTID,FST_ROADNAME,FST_SIFID,SEC_ROADNAME,SEC_SIFID,GEOMETRY
0,154647662,REHL RD,4976,W REHL CT,6856,"[-85.51044384084946, 38.20588660809318]"
1,254646551,REHL RD,4976,TUCKER STATION RD,5908,"[-85.52816882852979, 38.20037561255241]"
2,354646551,REHL RD,4976,TUCKER STATION RD,5908,"[-85.52841872335158, 38.200360349057064]"
3,431949996,I 64 EAST,3076,I 265 RAMP,8763,"[-85.50495414554669, 38.22260344055154]"
4,593499996,I 265 NORTH,8197,I 265 RAMP,8763,"[-85.50554642815675, 38.222127288727606]"


In [ ]:

# write transformed data to file.
store_path = "/Users/bencampbell/code/county_coverage/data/cleaner/intersections_data.json"

intersections_clean.to_json(store_path)

def read_in_intersections(path_to_intersection_json):
    out = pd.read_json(path_to_intersection_json)
    # fix some things on import
    out = out.set_index("INTID")
    out.GEOMETRY = out.GEOMETRY.apply(tuple)
    return out             

read_in_intersections(store_path).head()

#store_path_csv = "/Users/bencampbell/code/county_coverage/data/cleaner/intersections_data.csv"
#inter



,FST_ROADNAME,FST_SIFID,SEC_ROADNAME,SEC_SIFID,GEOMETRY
INTID,,,,,
5710837346,REHL RD,4976,W REHL CT,6856,"(-85.5104390301, 38.2058793146)"
10005800273,REHL RD,4976,TUCKER STATION RD,5908,"(-85.52814980550001, 38.2003487994)"
14300767569,REHL RD,4976,TUCKER STATION RD,5908,"(-85.5284139077, 38.2003530575)"
18011691414,I 64 EAST,3076,I 265 RAMP,8763,"(-85.5049493351, 38.2225961436)"
23945910678,I 265 NORTH,8197,I 265 RAMP,8763,"(-85.5055416176, 38.2221199919)"
